# Tiền xử lý merch/student sentiment

Notebook này đọc `../merch_datasets_student_sentiment.csv`, dùng cột `content_anonymized` làm raw input gốc và không ghi đè cột này. Nội dung sau xử lý được lưu vào cột mới `content_preprocessed`, rồi lưu file mới trong thư mục `Preprocessing`.

# 0. Cấu hình đường dẫn và import

In [ ]:
from pathlib import Path
import sys

import pandas as pd

cwd = Path.cwd()
if (cwd / "preprocess_student_sentiment.py").exists():
    PREPROCESSING_DIR = cwd
else:
    PREPROCESSING_DIR = (cwd / "STEP3_Labeling" / "Preprocessing").resolve()

if str(PREPROCESSING_DIR) not in sys.path:
    sys.path.insert(0, str(PREPROCESSING_DIR))

import preprocess_student_sentiment as prep

INPUT_CSV = prep.DEFAULT_INPUT_CSV
OUTPUT_CSV = prep.DEFAULT_OUTPUT_CSV
TEENCODE_PATH = prep.DEFAULT_TEENCODE_PATH
HASHTAG_MAP_PATH = prep.DEFAULT_HASHTAG_MAP_PATH
CONTENT_COLUMN = prep.CONTENT_COLUMN
OUTPUT_COLUMN = prep.OUTPUT_COLUMN

print("Input:", INPUT_CSV)
print("Output:", OUTPUT_CSV)
print("Teencode:", TEENCODE_PATH)
print("Hashtag map:", HASHTAG_MAP_PATH)
print("emoji package available:", prep.emoji_lib is not None)

# 1. Đọc dữ liệu gốc

In [ ]:
df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")

print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("Missing content:", int(df[CONTENT_COLUMN].isna().sum()))
df.head()

# 2. Nạp teencode và map hashtag cần loại bỏ

In [ ]:
resources = prep.load_resources(
    teencode_path=TEENCODE_PATH,
    hashtag_map_path=HASHTAG_MAP_PATH,
)

print("Teencode entries:", len(resources.teencode_map))
print("Hashtag remove regex:", len(resources.hashtag_remove_regex))
print("Hashtag remove normalized keys:", len(resources.hashtag_remove_normalized))

# 3. Kiểm tra pipeline trên vài ví dụ

In [ ]:
examples = [
    "#Cfs2295_DNTU Nhaaa k vui 😍 100k!!! https://abc.com",
    "#24399: mng đi 3 bước nhaaa :)))",
    "#thấygớm quá tr ơi 😡",
]

pd.DataFrame(
    {
        "raw": examples,
        "preprocessed": [prep.preprocess_text(text, resources) for text in examples],
    }
)

# 4. Chạy tiền xử lý cho cột content_anonymized

In [ ]:
df_processed = prep.preprocess_dataframe(
    df,
    resources=resources,
    content_column=CONTENT_COLUMN,
    output_column=OUTPUT_COLUMN,
    show_progress=True,
)

raw_column_unchanged = df_processed[CONTENT_COLUMN].equals(df[CONTENT_COLUMN])
print("Shape after preprocessing:", df_processed.shape)
print("Raw content_anonymized unchanged:", raw_column_unchanged)
assert raw_column_unchanged, "content_anonymized must stay identical to the source file"
df_processed[["record_key", CONTENT_COLUMN, OUTPUT_COLUMN]].head(10)

# 5. Lưu file mới

In [ ]:
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df_processed.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("Saved:", OUTPUT_CSV)
print("Rows:", len(df_processed))
print("Columns:", list(df_processed.columns))

# 6. Kiểm tra nhanh sau khi lưu

In [ ]:
df_check = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig")

expected_columns = list(df.columns) + [OUTPUT_COLUMN]
print("Output exists:", OUTPUT_CSV.exists())
print("Same row count:", len(df_check) == len(df))
print("Columns OK:", list(df_check.columns) == expected_columns)
print("Raw content_anonymized unchanged:", df_check[CONTENT_COLUMN].equals(df[CONTENT_COLUMN]))
assert df_check[CONTENT_COLUMN].equals(df[CONTENT_COLUMN]), "content_anonymized changed in output file"
print("Empty preprocessed rows:", int(df_check[OUTPUT_COLUMN].fillna("").str.len().eq(0).sum()))

df_check[["record_key", CONTENT_COLUMN, OUTPUT_COLUMN]].sample(10, random_state=42)